# Generate POS-tag asli (Stanza) untuk data NER Sirah

Mengisi kolom `pos_tag` (yang sekarang placeholder `NN`) dengan **UPOS riil** dari Stanza model `id`,
di-align 1:1 ke token yang sudah ada (pretokenized — TIDAK mengubah tokenisasi/baris/label).

**Langkah pakai:**
1. Runtime > Change runtime type > **GPU** (Stanza jauh lebih cepat; CPU juga jalan tapi lambat).
2. Upload `corrected_gold_data_20260610.zip` ke `MyDrive/TA-Sirah/` lalu extract di sana
   (atau pastikan `train.csv`, `test.csv`, `unlabelled.csv`, `train_augmented_v2.csv` sudah ada di folder itu).
3. Run all. Hasil: file di-overwrite dengan `pos_tag` terisi (+ backup `.bak_pos`), lalu di-zip & download.

> Catatan: data augmented = kalimat hasil swap (sintetik) → POS-nya tetap di-tag konsisten,
> sekadar sadar bahwa beberapa kalimat augmented bisa kurang gramatikal.

In [ ]:
!pip install -q stanza

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

DATASET_DIR = '/content/drive/MyDrive/TA-Sirah'
FILES = ['train.csv', 'test.csv', 'unlabelled.csv', 'train_augmented_v2.csv']

for f in FILES:
    p = os.path.join(DATASET_DIR, f)
    print(f"{f:<26}", 'OK' if os.path.exists(p) else 'MISSING', p)

In [ ]:
import stanza, torch
stanza.download('id', verbose=False)
USE_GPU = torch.cuda.is_available()
print('GPU:', USE_GPU)
# pretokenized = pakai token yang sudah ada (1 chunk = 1 'kalimat', tak di-split ulang)
nlp = stanza.Pipeline('id', processors='tokenize,pos',
                      tokenize_pretokenized=True, use_gpu=USE_GPU, verbose=False)

In [ ]:
import shutil
import pandas as pd
from collections import Counter

BATCH = 200  # chunk per panggilan Stanza

def _fit(upos, n):
    """Pastikan panjang upos == jumlah token chunk (potong / pad 'X')."""
    if len(upos) == n: return upos
    if len(upos) > n: return upos[:n]
    return upos + ['X'] * (n - len(upos))

def fill_pos(path, text_col='text_id', token_col='token', pos_col='pos_tag'):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    for need in (text_col, token_col):
        if need not in df.columns:
            print(f'  SKIP {os.path.basename(path)}: kolom {need} tak ada ({list(df.columns)})'); return
    if pos_col not in df.columns:
        df.insert(list(df.columns).index(token_col) + 1, pos_col, '')

    # group token per chunk (urutan baris dipertahankan)
    order, chunks = [], {}
    for tid, tok in zip(df[text_col], df[token_col]):
        if tid not in chunks:
            chunks[tid] = []; order.append(tid)
        t = str(tok)
        chunks[tid].append(t if t.strip() else '_')

    pos_map = {}
    for i in range(0, len(order), BATCH):
        bids = order[i:i + BATCH]
        batch = [chunks[t] for t in bids]
        doc = nlp(batch)
        if len(doc.sentences) == len(bids):
            for tid, sent in zip(bids, doc.sentences):
                pos_map[tid] = _fit([w.upos or 'X' for w in sent.words], len(chunks[tid]))
        else:  # fallback per-chunk kalau Stanza menggabung/memecah
            for tid in bids:
                d = nlp([chunks[tid]])
                upos = [w.upos or 'X' for s in d.sentences for w in s.words]
                pos_map[tid] = _fit(upos, len(chunks[tid]))
        print(f'  {min(i + BATCH, len(order))}/{len(order)} chunk', end='\r')

    its = {t: iter(pos_map[t]) for t in pos_map}
    df[pos_col] = [next(its[t]) for t in df[text_col]]

    bak = path + '.bak_pos'
    if not os.path.exists(bak):
        shutil.copy(path, bak)
    df.to_csv(path, index=False)
    print(f'\n  OK {os.path.basename(path)} | pos_tag top: {dict(Counter(df[pos_col]).most_common(6))}')

In [ ]:
for f in FILES:
    p = os.path.join(DATASET_DIR, f)
    if os.path.exists(p):
        print(f'=== {f} ===')
        fill_pos(p)

In [ ]:
# Verifikasi: pos_tag tidak lagi semua NN + contoh baris
import pandas as pd
df = pd.read_csv(os.path.join(DATASET_DIR, 'train.csv'), dtype=str, keep_default_na=False)
print('distribusi pos_tag train.csv:')
print(df['pos_tag'].value_counts())
print()
print(df.head(15).to_string(index=False))

In [ ]:
# Zip data ber-POS untuk di-download (extract ke repo lokal / re-bundle)
import zipfile
out = '/content/data_with_pos_20260610.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in FILES:
        p = os.path.join(DATASET_DIR, f)
        if os.path.exists(p):
            z.write(p, arcname=f)
print('zip:', out)
from google.colab import files
files.download(out)